# Setup

## Minimum setup for inference

In [ ]:
!git clone https://github.com/leducthanhig/machine-learning-project

In [ ]:
import os
os.chdir('machine-learning-project')

In [ ]:
!uv pip install -e . scipy

In [ ]:
import os

if os.path.exists('/kaggle/working'):
    from kaggle_secrets import UserSecretsClient # pyright: ignore[reportMissingImports]
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    os.environ['MANO_SESSID'] = user_secrets.get_secret("MANO_SESSID")
else:
    from google.colab import userdata # pyright: ignore[reportMissingImports]
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['MANO_SESSID'] = userdata.get('MANO_SESSID')

## Additional setup for visualization

### Install dependencies 

In [ ]:
!git submodule update --init --recursive

In [ ]:
!mkdir -p wheelhouse/pytorch3d_py312_torch2.3.0_cu121
!wget "https://drive.usercontent.google.com/download?id=1nX8GFmmbPhm6MiuLOqK-S29iQn4hVDkY&export=download&confirm=t" \
    -O wheelhouse/pytorch3d_py312_torch2.3.0_cu121/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl

In [ ]:
# Build PyTorch3D from source once, cache as a wheel, then reinstall from the cached wheel.
# This avoids a full rebuild on every new environment/kernel restart.
# Note: the wheel is ONLY valid for the same Python + PyTorch + CUDA combo used to build it.

!set -e && \
ENV_TAG=$(python -c "import sys, torch; cuda=(torch.version.cuda or 'cpu'); tv=torch.__version__.split('+')[0]; cu=cuda.replace('.', '') if cuda!='cpu' else 'cpu'; print(f'py{sys.version_info[0]}{sys.version_info[1]}_torch{tv}_cu{cu}')") && \
WHEELHOUSE="wheelhouse/pytorch3d_${ENV_TAG}" && \
mkdir -p "$WHEELHOUSE" && \
\
CUDA_HOME=$(python -c 'from torch.utils.cpp_extension import CUDA_HOME; print(CUDA_HOME or "")') && \
if [ -n "$CUDA_HOME" ] && [ -d "$CUDA_HOME/include/cub" ]; then \
  export CUB_HOME="$CUDA_HOME/include"; \
else \
  unset CUB_HOME; \
fi && \
\
echo "Using ENV_TAG=$ENV_TAG" && \
echo "Using WHEELHOUSE=$WHEELHOUSE" && \
echo "Using CUDA_HOME=$CUDA_HOME" && \
echo "Using CUB_HOME=${CUB_HOME-<unset>}" && \
\
if ls "$WHEELHOUSE"/pytorch3d-*.whl >/dev/null 2>&1; then \
  echo "Found cached PyTorch3D wheel(s) in $WHEELHOUSE"; \
else \
  echo "No cached wheel found; building PyTorch3D wheel (this can take a while)..." && \
  pip wheel -v --no-build-isolation --no-deps -w "$WHEELHOUSE" \
    "git+https://github.com/facebookresearch/pytorch3d.git@stable#egg=pytorch3d"; \
fi && \
\
uv pip install --no-build-isolation --no-deps --no-index --find-links "$WHEELHOUSE" pytorch3d

In [ ]:
!uv pip install --no-build-isolation smplx projectaria_tools \
    "chumpy @ git+https://github.com/mattloper/chumpy.git@master"

### Download model weights

In [ ]:
!curl 'https://download.is.tue.mpg.de/download.php?domain=mano&resume=1&sfile=mano_v1_2.zip' \
  -H 'Accept: text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8' \
  -H 'Accept-Language: en-US,en;q=0.9' \
  -H 'Accept-Encoding: gzip, deflate, br, zstd' \
  -H 'Connection: keep-alive' \
  -H 'Referer: https://mano.is.tue.mpg.de/' \
  -H 'Cookie: PHPSESSID={os.environ["MANO_SESSID"]}' \
  -o ./weights/mano/mano_v1_2.zip

In [ ]:
!unzip -j ./weights/mano/mano_v1_2.zip mano_v1_2/models/MANO_RIGHT.pkl \
    -d ./weights/mano/
!rm ./weights/mano/mano_v1_2.zip

In [ ]:
!wget https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/detector.pt \
    -P ./weights/hawor/external/
!wget https://huggingface.co/ThunderVVV/HaWoR/resolve/main/hawor/checkpoints/hawor.ckpt \
    -P ./weights/hawor/checkpoints/

# Run inference

In [ ]:
EXAMPLES = {
    '0001': {
        'file_name': '0001.jpg',
        'instruction': "Left: Put the trash into the garbage. Right: None.",
        'hand_arg': '--use_left',
    },
    '0002': {
        'file_name': '0002.jpg',
        'instruction': "Left hand: None. Right hand: Pick up the picture of Michael Jackson.",
        'hand_arg': '--use_right',
    },
    # This example requires more than 30GB RAM
    # '0003': {
    #     'file_name': '0003.png',
    #     'instruction': "Left hand: None. Right hand: Pick up the metal water cup.",
    #     'hand_arg': '--use_right',
    # },
}

In [ ]:
for ckpt_step_k in range(2, 21, 2):
    version = ckpt_step_k // 2
    for e in EXAMPLES:
        !python scripts/inference_human_prediction.py \
            --config "VITRA-VLA/VITRA-VLA-3B" \
            --model_path "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/{version}/final-epoch=0-step={ckpt_step_k}000.ckpt/weights.pt" \
            --image_path "./examples/{EXAMPLES[e]['file_name']}" \
            --sample_times 4 \
            --save_state_local \
            {EXAMPLES[e]['hand_arg']} \
            --video_path "./{e}_cp{ckpt_step_k}k.mp4" \
            --mano_path ./weights/mano \
            --instruction "{EXAMPLES[e]['instruction']}"

In [ ]:
for e in EXAMPLES:
    !python scripts/inference_human_prediction.py \
        --config "VITRA-VLA/VITRA-VLA-3B" \
        --model_path "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix/1/vitra-vla-3b.pt" \
        --image_path "./examples/{EXAMPLES[e]['file_name']}" \
        --sample_times 4 \
        --save_state_local \
        {EXAMPLES[e]['hand_arg']} \
        --video_path "./{e}_hf85k.mp4" \
        --mano_path ./weights/mano \
        --instruction "{EXAMPLES[e]['instruction']}"